# [프로젝트] 문서 요약 시스템 만들기

LLM을 사용하여, PDF 문서의 요약을 생성하는 어플리케이션을 구현해 보겠습니다.

1. Arxiv API에 검색어를 입력하여, 논문 정보를 가져옵니다.
2. Abstract와 Full Text를 불러옵니다. Full Text는 작은 크기로 청킹(Chunking)합니다.   
3. Step Back Prompting을 이용하여, Abstract의 내용을 바탕으로 요약의 포인트를 생성합니다. 
4. 요약 포인트를 이용하여 청크별 요약을 생성합니다.
5. 최종 요약본을 생성합니다.

In [1]:
!pip install openai arxiv langchain==0.3.27 langchain_community==0.3.27 langchain-openai langchain-google-genai langchain-community tiktoken pymupdf

  Using cached sgmllib3k-1.0.0-py3-none-any.whl
   ---------------------------------------- 0.0/18.4 MB ? eta -:--:--
   -- ------------------------------------- 1.3/18.4 MB 6.8 MB/s eta 0:00:03
   ----- ---------------------------------- 2.6/18.4 MB 6.7 MB/s eta 0:00:03
   -------- ------------------------------- 3.9/18.4 MB 6.6 MB/s eta 0:00:03
   ----------- ---------------------------- 5.2/18.4 MB 6.7 MB/s eta 0:00:02
   -------------- ------------------------- 6.8/18.4 MB 6.7 MB/s eta 0:00:02
   ----------------- ---------------------- 8.1/18.4 MB 6.7 MB/s eta 0:00:02
   -------------------- ------------------- 9.4/18.4 MB 6.7 MB/s eta 0:00:02
   ----------------------- ---------------- 10.7/18.4 MB 6.6 MB/s eta 0:00:02
   -------------------------- ------------- 12.1/18.4 MB 6.5 MB/s eta 0:00:01
   ----------------------------- ---------- 13.4/18.4 MB 6.4 MB/s eta 0:00:01
   ------------------------------- -------- 14.4/18.4 MB 6.3 MB/s eta 0:00:01
   ----------------------------

## 기본 LLM 불러오기

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    llm = ChatOpenAI(model='gpt-5-mini', temperature=1.0)
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, 
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

✅ GPT API 사용 가능!
✅ Gemini API 사용 가능!


필수 라이브러리를 불러옵니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


# Preliminary: LangChain의 Document Loader와 Text Splitter

LangChain은 외부 데이터와의 연결을 위한 다양한 도구를 제공합니다.

### 1) 랭체인 Document Loader

Document Loader는 파일 경로를 입력받아 데이터를 불러옵니다.

`langchain_community.document_loaders` (https://python.langchain.com/docs/integrations/document_loaders/)    

- CSVLoader, PyMuPDFLoader, WebBaseLoader 등의 다양한 로더가 존재합니다.

In [5]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader, CSVLoader
from langchain_community.document_loaders import ArxivLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.



로더는 load() 또는 비동기 방식으로 실행할 수 있으며, `Document` 클래스로 저장됩니다.
- metadata: Dict 형식으로 값이 저장됩니다.
- page_content: string 형태의 실제 문서 내용이 저장됩니다.


Data Loader와 Text Splitter   
이번 실습에서는 ArxivLoader를 사용합니다.

In [24]:
def load_arxiv_docs(query, max_chars=100000):
    loader = ArxivLoader(
        query=query, # 검색어
        load_max_docs=1, # 최대 문서 수 (여기서는 1개만 사용)
        doc_content_chars_max=max_chars, # 최대 문서 길이
        load_all_available_meta=True, # 모든 메타데이터 로드
    )
    return loader.load()


result = load_arxiv_docs("Korean Large Language Model Benchmark")[0]


print('전체 글자 수:', len(result.page_content))
print(result.metadata)

전체 글자 수: 59816
{'Published': '2024-12-10', 'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context', 'Authors': 'Xiaonan Wang, Jinyoung Yeo, Joon-Ho Lim, Hansaem Kim', 'Summary': "Large language models have exhibited significant enhancements in performance across various tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent content. Current multilingual benchmarks often use translated English versions, which may incorporate Western cultural biases that do not accurately assess other languages and cultures. To address this research gap, we introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning capabilities at the word, sentence, and paragraph levels. Using the KULTURE Bench, we assessed the capabilities of models trained w

In [25]:
result

Document(metadata={'Published': '2024-12-10', 'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context', 'Authors': 'Xiaonan Wang, Jinyoung Yeo, Joon-Ho Lim, Hansaem Kim', 'Summary': "Large language models have exhibited significant enhancements in performance across various tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent content. Current multilingual benchmarks often use translated English versions, which may incorporate Western cultural biases that do not accurately assess other languages and cultures. To address this research gap, we introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning capabilities at the word, sentence, and paragraph levels. Using the KULTURE Bench, we assessed the capabilities of models traine

### 2) 랭체인 Text Splitter

불러온 데이터는 랭체인의 `TextSplitter`를 통해 작은 단위로 분리합니다.

이를 청킹(Chunking)이라고 하는데, 이후의 RAG 파트에서 더 자세히 다룰 예정입니다.

In [26]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter: 문자나 토큰 단위로 분리하고, Overlap을 통해 내용 중첩
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10000,
    chunk_overlap=2000,
) # 0- 10000, 8000- 18000, 16000 -26000
# 구분자(delimiter, separator 기준으로) 청킹 (줄바꿈 기호, 공백, 마침표, 쉼표) 


chunks = text_splitter.split_documents([result])
print('Chunk 개수:', len(chunks))

Chunk 개수: 8


In [27]:
print(chunks[0].page_content)

Abstract 
Large language models (LLMs) have 
exhibited significant enhancements in 
performance across various tasks. However, 
the complexity of their evaluation increases 
as these models generate more fluent and 
coherent content. Current multilingual 
benchmarks often use translated English 
versions, which may incorporate Western 
cultural biases that do not accurately assess 
other languages and cultures. To address 
this research gap, we introduce KULTURE 
Bench, 
an 
evaluation 
framework 
specifically designed for Korean culture 
that features datasets of cultural news, 
idioms, and poetry. It is designed to assess 
language models' cultural comprehension 
and reasoning capabilities at the word, 
sentence, and paragraph levels. Using the 
KULTURE Bench, we assessed the 
capabilities of models trained with different 
language corpora and analyzed the results 
comprehensively. The results show that 
there 
is 
still 
significant 
room 
for 
improvement in the models’ understandi

이제, 텍스트 로드 결과를 활용하여 요약 시스템을 만들어 보세요!

In [28]:
# 1. Stuff - 1번

# 2. Map-Reduce - Chunk 개수 + 1 - 병렬 처리가 가능하기 때문에 빠름
# 3. Refine - Chunk 개수 - 토큰 소모는 제일 큼

# 4. Hybrid - 


## 2. Step-Back Prompting을 이용한 요약 가이드라인 생성 체인   

Step Back Prompting은 전체 답변을 작성하기 전 전체 구획이나 사전 정보를 준비하는 과정입니다.   
논문의 메타데이터를 입력하여 요약 가이드라인을 생성하는 체인을 구성하세요.    

-  LLM을 이용한 메타 프롬프팅을 추천합니다.

In [29]:
chunks[0].metadata

{'Published': '2024-12-10',
 'Title': 'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context',
 'Authors': 'Xiaonan Wang, Jinyoung Yeo, Joon-Ho Lim, Hansaem Kim',
 'Summary': "Large language models have exhibited significant enhancements in performance across various tasks. However, the complexity of their evaluation increases as these models generate more fluent and coherent content. Current multilingual benchmarks often use translated English versions, which may incorporate Western cultural biases that do not accurately assess other languages and cultures. To address this research gap, we introduce KULTURE Bench, an evaluation framework specifically designed for Korean culture that features datasets of cultural news, idioms, and poetry. It is designed to assess language models' cultural comprehension and reasoning capabilities at the word, sentence, and paragraph levels. Using the KULTURE Bench, we assessed the capabilities of models trained with differen

In [30]:
from langchain_core.prompts import ChatPromptTemplate


guideline_prompt = ChatPromptTemplate(
    [
        ('system','''
당신은 학술 논문 분석 전문가입니다.
         
논문의 제목과 Abstract을 분석하여, 논문을 요약할 때 중점적으로 다루어야 하는 핵심 기준을 도출해 주세요.
핵심 가이드라인을 개조식으로 작성하고.가이드라인만 출력하세요.
         
아래의 문제를 중점적으로 고려하세요.
         
1) 핵심 연구 질문(무슨 문제에 답하려고 하는가?)
2) 방법론 (어떤 접근법이 중요한가)
3) 주요 기여도(학술적으로 어떤 가치가 있는가?)
4) 평가 방식 (결과의 의미와 한계)
5) 기술적 세부사항

최종 가이드라인은 2000자 이내로 작성하세요.
'''),
        ('human','''
논문의 제목: {Title}
         
논문의 Abstract: {Summary}  
''')

    ]
)

guideliner = guideline_prompt | llm | StrOutputParser()

guide = guideliner.invoke(result.metadata)
# 매개변수에 해당하는 것만 전달

print(guide)


- 핵심 연구 질문: 논문이 답하려는 핵심 문제(한국 문화 맥락에서 LLM이 이해·추론할 수 있는지, 기존 다국어 벤치마크의 번역·서구 편향이 한국어 문화 평가에 적절한지)를 명확히 제시할 것.
- 평가 대상 및 범위: 평가가 다루는 문화 영역(뉴스, 관용구·속담, 시 등)과 언어 단위(단어·문장·문단 수준)를 분명히 구분하여 요약할 것.
- 동기와 문제 인식: 기존 다국어 벤치마크의 한계(번역 산출의 문화적 편향)와 한국 문화 전용 벤치의 필요성을 간결히 서술할 것.
- 데이터셋 구성 방법: 데이터 출처(코퍼스, 신문·문학 등), 수집 기준, 샘플 수, 레이블링·주석화 절차(어노테이터 구성, 합의 방법)를 포함해 요약할 것.
- 태스크·평가 설계: 각 수준별(단어/문장/문단) 구체적 태스크 유형(분류, 생성, 정답 선택, 의미 추론 등)과 입력·출력 형식을 명시할 것.
- 비교 대상 모델: 실험에 사용된 모델군(사전학습 언어별/코퍼스별 모델, 크기, 공개·비공개 여부)을 명확히 기술할 것.
- 평가 지표 및 절차: 사용한 정량적 지표(정확도, F1, BLEU/ROUGE 등), 평가 프로토콜(평가 데이터 분할, 반복 실험 여부, 통계적 유의성 검정)을 포함할 것.
- 분석 방법: 결과 분석의 범위(전반적 성능, 문화별·태스크별 성능 분해, 오류 유형 분석, 정성적 사례 제시)와 사용된 분석 기법을 언급할 것.
- 주요 기여도: 학술적·실용적 기여(한국 문화 전용 벤치 구축, 문화 이해·추론 평가 프레임워크, 공개 데이터/코드 제공 여부)를 명확히 요약할 것.
- 주요 발견과 한계: 모델들이 보인 핵심 결과(한국 문화의 심층적 이해에서 성능 한계)와 연구의 제한점(데이터 커버리지, 주석자 편향, 일반화 가능성)을 균형 있게 제시할 것.
- 재현성 및 자원 공개: 데이터셋·평가 스크립트·모델 체크포인트 공개 계획과 재현성을 위한 실험 설정(하이퍼파라미터, 하드웨어)을 포함할 것.
- 윤리적 고려사항: 문화적 민감성, 저작권, 주석자 안전 및 편향 완화 노력 등 윤

## 3. Map-Reduce 요약 프레임워크   

Map-Reduce 프레임워크는 Map-Reduce 과정으로 긴 문서를 요약하는 방식입니다.   
- Map: 전체 문서를 전체 문서를 부분으로 분리한 후, 각각 요약합니다.
- Reduce: 요약본을 모두 모아서, 최종 요약본을 작성합니다.

생성한 가이드라인을 이용해, 개별 문서를 요약하는 체인을 만들고 실행하세요.   

결과는 summaries에 리스트 형태로 저장하세요.

In [31]:
# Map 과정 : 각 문서에 대해 요약을 생성합니다.
from tqdm import tqdm
# 청크에 대한 요약 생성 (가이드라인 참고)

# 최종 요약만 한국어로 수행한다면 어떨까요?
map_prompt = ChatPromptTemplate(
    [
        ('system', '''주어진 논문의 내용을 요약하세요.
아래의 논문 일부와, 요약 가이드라인의 내용을 참고하여 상세한 요약문을 영어로 작성하세요.
요약문의 길이는 2-4개의 문단과 문단별 10문장 내외로 작성하세요.'''),
        ('human', '''
논문의 내용: {text}
         
---
         
요약 참고 가이드라인: {guide}''')
    ]
).partial(guide=guide)

map_chain  = map_prompt | llm | StrOutputParser()

summaries = await map_chain.abatch(chunks)
# 청크 개수만큼 LLM을 병렬 실행해서 요약문 생성
print(summaries)

['This paper addresses whether large language models (LLMs) can accurately understand and reason about Korean culture, and whether common multilingual evaluation practices—typically translations of English datasets with Western cultural priors—are suitable for evaluating non‑Western languages. To fill this gap the authors introduce KULTURE Bench, a culture‑specific evaluation suite for Korean containing 3,584 instances across three complementary datasets drawn from authentic sources (current news, commonly used idioms, and classical poems often taught in Korean textbooks). The benchmark is explicitly designed to probe cultural comprehension at three linguistic granularities: word level (KorID: 1,631 four‑character idioms in a cloze/multiple‑choice format), sentence level (KorPD: 453 poetic lines removed and to be selected from candidate lines), and paragraph level (KorCND: 1,500 culturally relevant news articles where models choose the correct headline among candidates after summarizin

In [32]:
summaries = map_chain.batch(chunks)

In [33]:
for summary in summaries:
    print(summary[0:100])
    print('-------------------')

This paper addresses whether large language models (LLMs) can accurately understand and reason about
-------------------
This paper introduces KULTURE Bench, a targeted evaluation framework designed to measure large langu
-------------------
This paper asks whether large language models (LLMs) can accurately understand and reason about Kore
-------------------
This paper asks whether large language models (LLMs) can accurately understand and reason about Kore
-------------------
This paper asks whether large language models (LLMs) can understand and reason about content embedde
-------------------
This paper introduces KULTURE Bench, a targeted evaluation framework and dataset suite designed to m
-------------------
This paper introduces KULTURE Bench, a benchmark designed to evaluate large language models’ underst
-------------------
This paper introduces KULTURE Bench, a culture-specific evaluation framework designed to measure lar
-------------------


summaries의 내용을 하나로 결합한 뒤, 전체 요약을 생성하는 Reduce 체인을 구성하세요.   

최종 결과는 summary에 저장하세요.

In [34]:
# Reduce 과정 : 각 문서의 요약을 하나로 합칩니다.
reduce_prompt = ChatPromptTemplate([
    ('system', '''논문 요약문의 리스트가 주어집니다.
이를 읽고, 전체 주제를 포함하는 최종 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.
     
형식은 크게 2단 목차로 구성된 마크다운으로 작성하세요.
     
답변은 한국어로 작성하세요.'''),
    ('human', '''

논문 요약문의 리스트: {summaries}

''')
])

summaries_str = '\n\n---\n\n'.join(summaries)
# (요약1 엔터 엔터 --- 엔터 엔터 요약2 ---)


reduce_chain = reduce_prompt | llm | StrOutputParser()

summary = reduce_chain.invoke(summaries_str)
print(summary)

# KULTURE Bench — 개요 요약

## 연구 질문과 동기
- 다국어 벤치마크들이 주로 영어 자료를 번역해 구성되면서 서구적 문화 편향을 내재할 가능성이 있다. 이러한 평가가 비(非)서구권 언어, 특히 한국어의 문화적·역사적 맥락을 제대로 검사하는지 의문이 제기된다.
- 본 논문은 “대형 언어모델(LLM)이 한국어 문화문맥(관용구·시·문화적 헤드라인 등)을 이해하고 추론할 수 있는가?”를 핵심 질문으로 삼고, 번역 기반 다국어 벤치가 놓치는 문화특이적 실패를 드러내기 위한 전용 평가체계 KULTURE Bench를 제안한다.

## 핵심 기여
1. 한국 문화에 초점을 맞춘 평가 스위트 KULTURE Bench를 제시 — 도메인(뉴스·관용구·시)과 언어 단위(단어/구·문장·문단)를 아우름.
2. 실제 한국어 원전(뉴스, 교과서 시, 사자성어 사전)에서 수집한 총 3,584개 인스턴스 공개 및 평가 파이프라인(예: OCR → 후보 선택 → 인간 검수) 제시.
3. 의미적으로 그럴듯한 오답(디스트랙터)을 KorBERT 임베딩과 코사인 유사도 빈을 이용해 자동/반자동으로 생성하는 재현 가능한 방법론 제공.
4. 다양한 사전학습 성향(한국어집중 모델 vs. 영어중심 모델 vs. 중국어중심 모델) 간 비교를 통해 문화적 근접성의 영향과 모델의 한계를 실증적으로 분석.
5. 데이터·코드 공개(논문·GitHub 링크 제공)와 함께 오류 유형 분류 및 질적 사례 분석을 제시하여 후속 연구를 유도.

# 데이터·방법·결과·한계

## 데이터셋 구성 요약
- 총 항목: 3,584개
  - KorID (관용구/사자성어): 1,631개(사전 OCR → 중복·오탈자 보정 → Modu/신문 코퍼스에서 문맥 추출 → 클로즈 테스트)
  - KorPD (시/구절): 453개(교과서 수록 시 OCR → 문장 샘플링(작품당 최대 5행) → 클로즈 항목)
  - KorCND (문화 뉴스/헤드라인 매칭): 1,500개(모두 수동 필터링으로 문화 관련 기사 선별 → 헤드라인-본문 정합성 테스

생성한 요약은 파일로 저장해도 좋습니다.

In [35]:
{result.metadata['Title']}

{'KULTURE Bench: A Benchmark for Assessing Language Model in Korean Cultural Context'}

In [36]:
with open(f'summary_{result.metadata['Title'][:4]}.md', 'w', encoding='utf-8') as f:
    f.write(summary)

<br><br><br><br><br><br>

### 성능 비교하기 (Future Work)

Stuff(전체를 입력하는 방식)과 비교하면 어떨까요?

In [37]:
prompt = ChatPromptTemplate([
    ('system', '''주어진 논문의 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.
답변은 한국어로 작성하세요.
'''),
    ('user', '''{text}
''')])

chain = prompt | llm | StrOutputParser()

original_summary = chain.invoke(result.page_content)

print(original_summary)

요약

본 논문은 한국 문화에 특화된 대형언어모델(LLM) 평가 프레임워크인 KULTURE Bench를 제안한다. 기존 다국어 벤치마크들이 영어 기반 자료의 번역에 의존하면서 서구 문화 편향을 반영하는 반면, KULTURE Bench는 한국어와 한국 문화에 내재된 표현·맥락을 직접 평가하도록 설계되었다. 전체 데이터는 3,584개 항목으로 구성되며, 단어·문장·문단 수준에서 문화적 이해력과 추론 능력을 측정하는 세 가지 데이터셋으로 이루어져 있다.

데이터셋 구성
- KorID (1,631개): 한자성어·사자성어(네 글자 관용구)를 대상으로 하는 클로즈 테스트. 어휘는 기존 사전(예: Gosa, Korean Four-Character Idiom Grand Dictionary)에서 수집하고 OCR·수정 과정을 거쳐 5,372개 고유 이디엄을 확보한 뒤, Modu Corpus(신문 코퍼스)에서 실제 사용 문맥을 추출하여 구성했다. 오답 후보는 ETRI의 KorBERT 임베딩을 이용해 코사인 유사도 구간([0.5–0.6], [0.61–0.7], [0.71–0.8], [0.81–0.9])에서 각 1개씩 뽑아 생성했다. 유의미한 동의어 혼동을 피하기 위해 유사도 0.9 이상은 제외했고, 무작위 샘플 검토로 골든 정답의 선택성이 확인되었다.
- KorPD (453개): 교과서에 수록된 한국 시(총 91편)에서 임의의 행을 공란으로 만들어 문장 수준의 이해(의미·운율·수사 등)를 평가하는 클로즈 테스트로 구성. 각 행의 의미 표현은 KorBERT의 [CLS] 임베딩을 사용해 유사도 기반 오답 후보를 선정했다.
- KorCND (1,500개): 문화 관련 신문 기사 요약–헤드라인 매칭 과제. Modu Corpus(2021년 NIKL 신문)에서 “문화” 관련 기사를 수집·검수하여 마련했으며, 기사 제목의 [CLS] 임베딩 기반 유사도 구간에서 오답 후보를 선택, 골든 타이틀과 유사하지만 미세하게 다른 선택지를 제공해 문단 수준의 이해력을 검증한다.

실험 설정
평가 대상 모델

Stuff와 Step-Back만 수행한 경우와도 비교해 보겠습니다.

In [38]:
prompt = ChatPromptTemplate([
    ('system', '''주어진 논문의 요약을 한국어로 작성하세요.
요약은 8000자 이내로 작성하세요.
답변은 한국어로 작성하세요.
'''),
    ('user', '''{text}
---

다음은 요약 가이드라인입니다.

{guide}''')]).partial(guide=guide)

chain = prompt | llm | StrOutputParser()

stepback_summary = chain.invoke(result.page_content)

print(stepback_summary)

요약 (한국어)

핵심 연구 질문
- 본 논문은 “한국 문화 맥락에서 대형언어모델(LLM)이 문화적 이해 및 추론을 제대로 수행할 수 있는가?”와 “기존 다국어 벤치마크의 영어 원문 번역은 서구 문화 편향으로 인해 한국어·한국 문화 평가에 적절한가?”라는 문제를 다룬다. 이를 위해 한국 문화에 특화된 평가 프레임워크 KULTURE Bench를 제안하고, 서로 다른 언어 코퍼스로 학습된 모델들이 한국 문화 텍스트(관용구·시·문화뉴스)를 얼마나 잘 처리하는지 실험·분석한다.

평가 대상·범위 및 동기
- 평가 영역: 한국 문화 관련 텍스트(관용구/사자성어, 고전·교과서 수록 시, 한국 문화 뉴스).
- 언어 단위: 단어 수준(idiom cloze; KorID), 문장 수준(poem line cloze; KorPD), 문단/문서 수준(news summarization/headline matching; KorCND).
- 문제 인식: 기존 다국어 벤치마크는 영어 기반 데이터의 번역에 의존하는 경우가 많아 서구 문화 편향을 내포하며, 이는 한국 고유의 문화적 맥락을 평가하는 데 한계가 있으므로 한국 문화 전용 벤치가 필요함.

데이터셋 구성(출처·수집·샘플 수·주석)
- 전체 규모: 총 3,584개 인스턴스. KorID 1,631, KorPD 453, KorCND 1,500.
- 출처:
  - KorID: 고사·사자성어 사전(Gosa, Han)에서 사자성어 어휘 수집(5,372개), Modu Corpus(특히 NIKL 신문 코퍼스, 2009–2022)에서 해당 관용구가 등장하는 뉴스 발췌.
  - KorPD: 교과서에 수록된 시 91편을 OCR로 디지털화·검수하여 라인 단위로 샘플링(총 453개 인스턴스).
  - KorCND: Modu Corpus의 NIKL Newspaper Corpus(2021년 기사 집합)에서 '문화' 관련 키워드 기반 자동 필터링 후 수동 검토로 선별한 문화 기사 1,500건.
- 전처리·어노테이션:
  - OCR 후 수동 검수로 오탈자·메타데

과거에는 요약 성능을 평가하기 위해 레퍼런스 요약문과 실제 요약을 키워드로 비교하는 ROUGE 등의 Score를 사용했으나,   

최근에는 LLM을 통해 평가하는 경우도 많습니다.

In [40]:
# 다양한 평가 기준을 세우고, Structured Output을 통해 평가하는 체인을 구성해도 좋습니다!

evaluation_prompt = ChatPromptTemplate(
    [
        ('system', '''논문의 원본 내용과, 이를 3가지 방법으로 요약한 요약문이 주어집니다.
세 요약문의 순위를 매기고, 그 이유를 설명하세요.

평가 기준은 다음과 같습니다.

- 논문에서 풀고자 하는 문제에 대해 정확히 파악했는가?
- 원문의 핵심 내용과 시사점을 잘 포함하고 있는가?
- 연구의 제약점과 향후 발전 방향에 대해 잘 기술하고 있는가?

'''),
        ('user', '''
원본 내용: 
{original_text}


---

1번 요약문: 
{summary1}

---

2번 요약문: 
{summary2}

---

3번 요약문: 
{summary3}
''')
])

evaluation_chain = evaluation_prompt | llm | StrOutputParser()

evaluation_result = evaluation_chain.invoke({
    'original_text': result.page_content,
    'summary1': summary,
    'summary2': stepback_summary,
    'summary3': original_summary
})

print(evaluation_result)

요약문들에 대한 순위(1→3)와 이유는 다음과 같습니다.

1위 — 1번 요약문
- 문제 파악: 논문이 다루는 핵심 질문(영어 번역 기반 벤치의 문화적 편향 문제 및 한국 문화 특화 평가 필요성)을 명확하고 정확하게 제시했습니다.
- 핵심 내용·시사점 포함 여부: 데이터셋(KorID/KorPD/KorCND)의 구성·규모·수집·후보 생성 방법(KorBERT 임베딩·코사인 유사도 빈)을 구체적으로 재현했고, 실험 설계(모델군·제로샷·CoT·추론 길이)와 주요 수치(모델별 정확도, CoT 효과)까지 잘 요약했습니다. 오류 유형 분석과 질적 사례, GitHub 공개 등 논문의 실무적 기여도도 빠짐없이 담았습니다.
- 제약점·향후 방향 기술: 재현성·표본 크기·평가 범위 등의 한계를 명확히 지적하고 윤리·저작권, 향후 멀티모달·교차문화 확장 등 구체적 권고까지 제시했습니다.
- 종합평: 가장 완전하고 균형 잡힌 요약입니다. (약간 길지만 논문 핵심과 시사점·한계를 충실히 반영)

2위 — 2번 요약문
- 문제 파악: 핵심 문제와 동기는 명확히 짚었습니다.
- 핵심 내용·시사점 포함 여부: 데이터 출처·전처리·디스트랙터 생성 방법, 실험 프로토콜(응답 검증 규칙 포함), 주요 결과(수치 포함), 인간 비교, 오류 분류를 충실히 다루었습니다. 1번과 매우 유사한 수준의 상세도를 갖고 있습니다.
- 제약점·향후 방향 기술: 재현성·샘플 크기·모델군 한계, 윤리적 고려 등 적절히 기술했습니다. 통계적 유의성 표기 누락 등 재현성 관점에서의 언급도 포함해 실무적 관점에서 장점이 있습니다.
- 약점: 1번과 비교하면 일부 세부(예: 동의어 검증 절차의 구체적 점수 기준 등)가 덜 상세하고, 전반적 구성은 1번에 비해 약간 덜 체계적입니다.
- 종합평: 매우 competent한 요약으로 세부적 비판까지 잘 담았으나 1번이 더 포괄적이라 2위.

3위 — 3번 요약문
- 문제 파악: 핵심 질문과 연구 동기는 제대로 파악되어 있습니다.
- 핵심 내용·시사점 포함 여부: 데이터셋 구성·